# UCS420: Cognitive Computing
## Assignment 7 — NLP-I

**Name:**Divyanshi Dhawan **Roll No:** 1024160057

> **IMPORTANT — how to use this notebook**
> The assignment says to use the dataset from `UCS420_LA9_Starter.ipynb`. I could not open that private Colab link, so **Section 0 contains a stand-in** version of `generate_dataset()`, `process_ticket()` and `TEST_LINES` so that everything runs end-to-end.
> **Before submitting:** replace Section 0's *stand-in cell* with the original starter code (copy the cells from your starter notebook that define `generate_dataset`, `process_ticket`, `TEST_LINES`, and print the fingerprint). Then edit the small `get_parts()` adapter if your `process_ticket()` returns a different structure. Everything else is written to work on top of that adapter.

## 0. Setup

In [ ]:
import re, string, collections, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import nltk
for pkg in ['punkt', 'punkt_tab', 'stopwords', 'wordnet', 'omw-1.4',
            'averaged_perceptron_tagger', 'averaged_perceptron_tagger_eng']:
    nltk.download(pkg, quiet=True)
from nltk.tokenize import sent_tokenize, word_tokenize, RegexpTokenizer
from nltk.corpus import stopwords, wordnet
from nltk.stem import PorterStemmer, LancasterStemmer, RegexpStemmer, WordNetLemmatizer
pd.set_option('display.max_columns', None); pd.set_option('display.width', 200)

### 0.1 STAND-IN dataset — **REPLACE this cell with your original starter code**

In [ ]:
# ====================  STAND-IN (replace with the starter notebook's code)  ====================
import random, hashlib
ROLL_NO = "102303001"          # <-- enter your roll number

def generate_dataset(roll_no=ROLL_NO):
    rnd = random.Random(int(hashlib.md5(str(roll_no).encode()).hexdigest(), 16) % (10**8))
    templates = [
        "My laptop isn't connecting to the Wi-Fi since the update. Please call me on 555-123-4567 or email john.doe@example.com.",
        "The VPN client version 2.0.1 keeps crashing. I contacted Support at https://help.example.com/vpn yesterday!",
        "Printer on Floor 3 is printing blank pages. Battery shows 95.5% but the toner is empty.",
        "Outlook sends error 0x80041 when I open attachments. Reach me at +1 (555) 987-6543 or anita_k@corp.org.",
        "Our state-of-the-art monitor flickers randomly. We're using driver 4.12.3 on Windows 11, and it isn't fixed.",
        "Application crashed after installing patch 1.5. Is there an update? Visit www.fixit-now.com for details.",
        "Cannot access the shared drive; permissions were changed. Admin said the server runs at 99.9% uptime.",
        "Email synchronization fails on my phone. Contact: it-help@university.edu. Error appears every morning.",
        "Keyboard keys are sticking and the touchpad isn't responding. Ordered a replacement on https://shop.example.com/kb.",
        "Wi-Fi drops every 30 minutes in Building A. Signal strength is 62.5 dBm; please escalate to Network team.",
        "Software licence expired yesterday. Please renew version 3.1.4 and call 98765-43210 for confirmation.",
        "Unable to reset my password. The portal shows an error, and I'm locked out of every account!",
        "Backup is running slowly; transferring 120.5 GB takes hours. Is it a configuration issue?",
        "Projector in Room 204 shows a purple tint. Technicians already replaced the cable, but it isn't solved.",
        "Antivirus scan reports 3 threats. Updated definitions to 7.0.2 and rebooted; email security@example.com if needed.",
        "Installation of Adobe Reader fails with code 1603. Everything else works, however the installer hangs at 45%.",
    ]
    n = rnd.randint(12, 16)
    return rnd.sample(templates, k=min(n, len(templates)))

STOP = set(stopwords.words('english'))
LEMM = WordNetLemmatizer()

def process_ticket(text):
    sents  = sent_tokenize(text)
    tokens = word_tokenize(text)
    punct  = [t for t in tokens if all(ch in string.punctuation for ch in t)]
    stops  = [t for t in tokens if t.lower() in STOP]
    final  = [t.lower() for t in tokens
              if t not in punct and t.lower() not in STOP and any(ch.isalnum() for ch in t)
              and not t.startswith("'") and t.lower() != "n't"]
    return {"sentences": sents, "tokens": tokens, "punct": punct, "stopwords": stops,
            "unique": sorted(set(tokens)), "final": final}

TEST_LINES = [
    "Contact me at john.doe@example.com or call +1 (555) 123-4567 today.",
    "Visit https://www.example.com/path?x=1 or www.fixit-now.com for help.",
    "It isn't working: the Wi-Fi speed is 95.5 Mbps on version 2.0.1.",
    "This state-of-the-art router, called Alpha, costs 1299.99 dollars.",
    "Our Engineers' updates aren't ready; Everything is under review.",
]
# ==============================================================================================

tickets = generate_dataset(ROLL_NO)
fingerprint = hashlib.md5("".join(tickets).encode()).hexdigest()[:8]
print("Number of tickets:", len(tickets), "| fingerprint:", fingerprint)
for i, t in enumerate(tickets, 1):
    print(f"T{i:02d}: {t}")

### 0.2 Adapter (edit only if your real `process_ticket()` returns something different)

In [ ]:
def get_parts(ticket):
    """Return a dict with: sentences, tokens, punct, stopwords, unique, final  (all lists)."""
    r = process_ticket(ticket)
    return {
        "sentences": r["sentences"],
        "tokens":    r["tokens"],
        "punct":     r["punct"],
        "stopwords": r["stopwords"],
        "unique":    r["unique"],
        "final":     r["final"],
    }

parts = [get_parts(t) for t in tickets]

## 1. Preprocessing

In [ ]:
rows = []
for i, p in enumerate(parts, 1):
    rows.append({
        "Ticket": f"T{i:02d}",
        "Sentences":     len(p["sentences"]),
        "Tokens":        len(p["tokens"]),
        "Punct. tokens": len(p["punct"]),
        "Stopwords":     len(p["stopwords"]),
        "Unique tokens": len(set(p["unique"])),
        "Final tokens":  len(p["final"]),
    })
df1 = pd.DataFrame(rows).set_index("Ticket")
total = df1.sum(numeric_only=True)
df1.loc["TOTAL"] = total
df1

*Note:* the TOTAL row of **Unique tokens** is a plain sum of per-ticket values (it double counts words that appear in several tickets). The true corpus-level unique count is calculated in Section 5.

**Detailed view of one selected ticket**

In [ ]:
SEL = 0     # <-- index of the ticket to display (0 = T01)
p = parts[SEL]
print(f"Ticket T{SEL+1:02d}: {tickets[SEL]}\n")
print("Tokens        :", p["tokens"])
print("Punctuation   :", p["punct"])
print("Stopwords     :", p["stopwords"])
print("Final tokens  :", p["final"])

## 2. Regex extraction and replacement (raw tickets, `re`)

In [ ]:
raw_text = "\n".join(tickets)

PAT_LONG   = r"\b[A-Za-z]{6,}\b"                    # words longer than 5 letters
PAT_NUM    = r"(?<![\w.])\d+(?:\.\d+)*(?![\w])"      # 95.5, 2.0.1 stay as ONE match
PAT_CAP    = r"\b[A-Z][a-z]*\b"                      # capitalised words
PAT_VOWEL  = r"\b[AEIOUaeiou][A-Za-z]*\b"            # words beginning with a vowel

def extract_all(text):
    return {
        "long words (>5 letters)": re.findall(PAT_LONG, text),
        "numbers":                 re.findall(PAT_NUM, text),
        "capitalised words":       re.findall(PAT_CAP, text),
        "vowel-initial words":     re.findall(PAT_VOWEL, text),
    }

for k, v in extract_all(raw_text).items():
    print(f"{k} ({len(v)}):\n   {v}\n")

In [ ]:
# --- Replacement patterns (order matters: email -> URL -> phone) ---
PAT_EMAIL = r"[\w.+-]+@[\w-]+(?:\.[\w-]+)+"
PAT_URL   = r"(?:https?://|www\.)[^\s<>\"']*[\w/#=&%-]"
PAT_PHONE = r"(?<![\w.])(?:\+\d{1,3}[\s.-]?)?(?:\(\d{3}\)|\d{3})[\s.-]?\d{3}[\s.-]?\d{4}(?!\d)|(?<![\w.])\d{5}[\s.-]\d{5}(?!\d)"

def mask(text):
    text = re.sub(PAT_EMAIL, "<EMAIL>", text)
    text = re.sub(PAT_URL,   "<URL>",   text)
    text = re.sub(PAT_PHONE, "<PHONE>", text)
    return text

masked = [mask(t) for t in tickets]
for a, b in zip(tickets, masked):
    if a != b:
        print("RAW   :", a); print("MASKED:", b); print()

**Check the patterns on `TEST_LINES`**

In [ ]:
for line in TEST_LINES:
    print("LINE   :", line)
    for k, v in extract_all(line).items():
        print(f"   {k:25s}: {v}")
    print("   MASKED :", mask(line)); print()

## 3. Custom tokenization — `my_tokenize(text)` with one `RegexpTokenizer` pattern

In [ ]:
TOKEN_PATTERN = r"""
    [\w.+-]+@[\w-]+(?:\.[\w-]+)+                                  # e-mail
  | (?:https?://|www\.)[^\s<>"']*[\w/#=&%-]                        # URL
  | (?:\+\d{1,3}[\s.-]?)?(?:\(\d{3}\)|\d{3})[\s.-]\d{3}[\s.-]\d{4}  # phone  (555-123-4567, +1 (555) 987-6543)
  | \d{5}[\s.-]\d{5}                                              # phone  (98765-43210)
  | \d+(?:\.\d+)+                                                # decimals / versions (95.5, 2.0.1)
  | \w+(?:[-']\w+)*                                               # words, contractions (isn't), hyphenated (Wi-Fi)
  | [^\w\s]                                                       # any other single punctuation mark
"""
_tok = RegexpTokenizer(TOKEN_PATTERN, flags=re.VERBOSE | re.UNICODE)

def my_tokenize(text):
    return _tok.tokenize(text)

print("my_tokenize on TEST_LINES\n")
for line in TEST_LINES:
    print(line); print("  ->", my_tokenize(line)); print()

In [ ]:
print("my_tokenize on tickets\n")
for i, t in enumerate(tickets, 1):
    print(f"T{i:02d}:", my_tokenize(t))

### Comparison with `word_tokenize()`

In [ ]:
def compare(text):
    a, b = my_tokenize(text), word_tokenize(text)
    only_mine = [x for x in a if x not in b]
    only_nltk = [x for x in b if x not in a]
    return a, b, only_mine, only_nltk

diff_rows = []
for src, texts in [("TEST", TEST_LINES), ("TICKET", tickets)]:
    for i, t in enumerate(texts, 1):
        a, b, om, on = compare(t)
        if om or on:
            diff_rows.append({"Source": f"{src}{i}", "Only my_tokenize": om, "Only word_tokenize": on})
diff_df = pd.DataFrame(diff_rows)
pd.set_option('display.max_colwidth', 120)
diff_df

In [ ]:
# Concrete examples for the write-up (taken from the actual outputs)
for line in TEST_LINES[:4]:
    print(line)
    print("  mine :", my_tokenize(line))
    print("  nltk :", word_tokenize(line)); print()

### Differences identified (confirm against the diff table above — it is generated from your data)

1. **E-mail addresses** — `my_tokenize` keeps `john.doe@example.com` as **one** token; `word_tokenize` breaks it into `john.doe`, `@`, `example.com`, because it treats `@` as a separate symbol.
2. **URLs** — `my_tokenize` keeps `https://www.example.com/path?x=1` whole; `word_tokenize` fragments it into `https`, `:`, `//www.example.com/path`, `?`, `x=1`. (Bare `www.fixit-now.com` is usually left intact by both.)
3. **Phone numbers** — `my_tokenize` keeps `+1 (555) 123-4567` as one token; `word_tokenize` gives `+1`, `(`, `555`, `)`, `123-4567`.
4. **Contractions** — `my_tokenize` keeps `isn't`, `aren't`, `I'm`, `We're` whole; `word_tokenize` follows Penn-Treebank rules and splits them into `is` + `n't`, `I` + `'m`, `We` + `'re`.
5. **`Cannot`** — `word_tokenize` splits it into `Can` + `not` (a Treebank special case); `my_tokenize` keeps `Cannot` as written.
6. **Same on both:** hyphenated words (`Wi-Fi`, `state-of-the-art`), decimals (`95.5`) and versions (`2.0.1`) come out as single tokens in both tokenizers.

*Why:* `word_tokenize` follows Penn-Treebank conventions designed for syntactic parsing (split clitics, separate punctuation symbols), whereas `my_tokenize` is pattern-driven and preserves "entity-like" strings (e-mail, URL, phone) that matter for IT-support ticket analysis.

## 4. Stemming and lemmatization on the unique final tokens

In [ ]:
all_final = [w for p in parts for w in p["final"]]
unique_final = sorted(set(all_final))
print(len(unique_final), "unique final tokens")
print(unique_final)

**Choosing a suffix for the Regexp Stemmer** — look at which candidate suffixes actually occur in this dataset

In [ ]:
candidates = ["ing", "ed", "ly", "es", "s", "ion", "er"]
suffix_examples = {s: [w for w in unique_final if w.endswith(s) and len(w) > len(s) + 2] for s in candidates}
for s, ws in sorted(suffix_examples.items(), key=lambda kv: -len(kv[1])):
    print(f"-{s:4s}: {len(ws):2d} words  e.g. {ws[:6]}")

In [ ]:
# Pick the suffix pattern that matches the most dataset words
best = sorted(suffix_examples, key=lambda s: -len(suffix_examples[s]))[:2]
SUFFIX_PATTERN = "|".join(f"{s}$" for s in best)       # e.g. 'ing$|s$'
print("Chosen Regexp-stemmer pattern:", SUFFIX_PATTERN)
for s in best:
    print(f"  suffix '-{s}' justified by dataset words: {suffix_examples[s][:4]}")

**Justification:** the suffixes above were chosen because they occur most frequently in this dataset's final tokens (examples are printed above, e.g. verb forms such as *-ing* and plural/3rd-person *-s*). Stripping them merges inflected variants (e.g. *crashes/crashed/crashing → crash*) without needing a dictionary. `min=4` prevents very short words (e.g. *vpn*, *bus*) from being over-stripped.

In [ ]:
porter    = PorterStemmer()
lancaster = LancasterStemmer()
regexp    = RegexpStemmer(SUFFIX_PATTERN, min=4)
lemm      = WordNetLemmatizer()

def to_wn(tag):
    return {'J': wordnet.ADJ, 'V': wordnet.VERB, 'N': wordnet.NOUN, 'R': wordnet.ADV}.get(tag[0], wordnet.NOUN)

tagged = nltk.pos_tag(unique_final)
tag_of = dict(tagged)

res = pd.DataFrame({
    "Word": unique_final,
    "POS": [tag_of[w] for w in unique_final],
    "Porter": [porter.stem(w) for w in unique_final],
    "Lancaster": [lancaster.stem(w) for w in unique_final],
    "Regexp": [regexp.stem(w) for w in unique_final],
    "WordNet Lemma (POS)": [lemm.lemmatize(w, to_wn(tag_of[w])) for w in unique_final],
})
diff_only = res[res[["Porter", "Lancaster", "Regexp", "WordNet Lemma (POS)"]].nunique(axis=1) > 1]
print(f"{len(diff_only)} of {len(res)} words give different outputs")
diff_only.reset_index(drop=True)

## 5. Corpus statistics

In [ ]:
S1 = [t for p in parts for t in p["tokens"]]                                   # all tokens
S2 = [t for p in parts for t in p["tokens"] if t not in set(p["punct"])]       # no punctuation
S3 = [w for p in parts for w in p["final"]]                                    # final tokens
S4 = [porter.stem(w) for w in S3]                                              # S3 + Porter

sets = {"S1 (all tokens)": S1, "S2 (no punct.)": S2, "S3 (final)": S3, "S4 (S3 + Porter)": S4}
stats = pd.DataFrame({
    name: {"Total tokens": len(s), "Unique tokens": len(set(s)), "TTR": round(len(set(s)) / len(s), 4)}
    for name, s in sets.items()
}).T
stats

In [ ]:
print("Top 10 words — S1 (all tokens):")
print(collections.Counter(S1).most_common(10))
print("\nTop 10 words — S3 (final tokens):")
print(collections.Counter(S3).most_common(10))

**Zipf plot — log(rank) vs log(frequency) for S3**

In [ ]:
freq = sorted(collections.Counter(S3).values(), reverse=True)
ranks = np.arange(1, len(freq) + 1)
plt.figure(figsize=(6, 4))
plt.plot(np.log(ranks), np.log(freq), 'o-', ms=4)
plt.xlabel("log(rank)"); plt.ylabel("log(frequency)")
plt.title("Zipf plot of S3 (final tokens)")
plt.grid(alpha=.3); plt.show()
slope = np.polyfit(np.log(ranks), np.log(freq), 1)[0]
print(f"Fitted slope ≈ {slope:.2f}  (ideal Zipf ≈ -1; small corpora deviate because of many words occurring once)")

In [ ]:
from nltk import bigrams
print("Top 5 bigrams — S1:", collections.Counter(bigrams(S1)).most_common(5))
print("Top 5 bigrams — S3:", collections.Counter(bigrams(S3)).most_common(5))
print("\nNote: bigrams are computed over the concatenated token stream, so a few pairs span ticket boundaries.")

## 6. Interpretation

**(a) Why does TTR change from S1 to S4?**

In [ ]:
for name, s in sets.items():
    print(f"{name:18s} total={len(s):4d} unique={len(set(s)):4d} TTR={len(set(s))/len(s):.3f}")

TTR = unique tokens / total tokens. Going **S1 → S2** removes punctuation, which is highly repetitive (`.`, `,`), so total tokens drop a lot while unique tokens barely change → TTR **rises**. Going **S2 → S3** removes stopwords (*the, is, to…*), the most repeated words, so the total falls sharply while most unique content words stay → TTR rises again. Going **S3 → S4** (Porter stemming) merges inflected variants (e.g. *crashes/crashed → crash*), so the number of unique types falls while the total stays the same → TTR **drops slightly** compared with S3. Overall, TTR is highest for S3 and a bit lower for S4. Check the table above for the exact numbers on your dataset.

**(b) Example where `word_tokenize()` and my tokenizer differ**

In [ ]:
ex = next((t for t in TEST_LINES + tickets if "isn't" in t), TEST_LINES[2])
print(ex); print("word_tokenize:", word_tokenize(ex)); print("my_tokenize  :", my_tokenize(ex))

`word_tokenize` splits **isn't → `is`, `n't`** (Treebank convention, useful for syntax), while `my_tokenize` keeps **`isn't`** as one token. The same happens with URLs and phone numbers, which `word_tokenize` fragments and `my_tokenize` preserves.

**(c) Example where stemming gives an undesirable result**

In [ ]:
bad = []
for w in unique_final:
    st = porter.stem(w)
    if st != w and not wordnet.synsets(st):          # stem is not a real English word
        bad.append((w, st, lancaster.stem(w)))
print("Words whose Porter stem is not a dictionary word (word, Porter, Lancaster):")
for b in bad[:10]: print("  ", b)

In [ ]:
examples = ", ".join(f"{w} -> {p}" for w, p, _ in bad[:4])
print(f"From this dataset, Porter produces non-words such as: {examples}")

Stemmers chop suffixes with fixed rules and do not check a dictionary, so they can create **non-words** (the examples printed above, such as *application → applic* or *already → alreadi*) and can **over-stem**, merging words with different meanings or producing stems that are hard to read. Lancaster is even more aggressive (see its column in the table above). Use one of the printed examples from your own dataset in your answer.

**(d) POS-aware lemmatization preserving meaning better than stemming**

In [ ]:
demo = ["updates", "crashing", "installed", "better", "running", "was"]
print(f"{'word':10s} {'POS':5s} {'Porter':10s} {'Lemma(POS-aware)':18s}")
for w in demo:
    tag = nltk.pos_tag([w])[0][1]
    print(f"{w:10s} {tag:5s} {porter.stem(w):10s} {lemm.lemmatize(w, to_wn(tag)):18s}")
print()
print("lemmatize('better', 'a') =", lemm.lemmatize('better', 'a'), "| Porter('better') =", porter.stem('better'))
print("lemmatize('was',    'v') =", lemm.lemmatize('was', 'v'),    "| Porter('was')    =", porter.stem('was'))

The POS-aware lemmatizer returns real dictionary words: **was → be** (verb) where Porter gives the non-word **wa**, **updates → update**, and **better → good** when it is tagged as an adjective (`lemmatize('better','a')`; note the automatic tag in the table may differ, e.g. RBR gives *well*, which shows why correct POS information matters). Because it uses the POS tag it can also separate the noun *saw* (a tool) from the verb *saw* (past of *see*), something a rule-based stemmer cannot do.